# 01 — Data quality, field mapping and capabilities

This notebook uses the same data services as the AutoDS application. It loads the labelled demo dataset, inspects data quality, shows how source columns map to analytical fields, and verifies which analyses the dataset can support.

The demo data is synthetic and exists only for exploration. Replace the dataset id in the setup cell to inspect an uploaded dataset.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "backend").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "backend"))

import pandas as pd

from app.database.base import SessionLocal, init_db
from app.services.data_service import ensure_demo_dataset, get_context
from app.services.mapping import capabilities_view, field_mapping_view

init_db()
db = SessionLocal()
dataset = ensure_demo_dataset(db)
ctx = get_context(dataset.id, db)
print(f"Dataset: {dataset.name}")
print(f"Rows: {dataset.rows:,} | Columns: {dataset.columns} | Demo: {dataset.is_demo}")

## Quality profile

Quality is calculated from missing cells, duplicate rows and invalid dates. Outliers are reported but are not automatically deleted.

In [ ]:
quality = dataset.quality_report
quality_summary = {
    "quality_score": quality["quality_score"],
    "rows": quality["rows"],
    "columns": quality["columns"],
    "missing_pct": quality["missing_pct"],
    "duplicate_rows": quality["duplicate_rows"],
    "invalid_dates": quality["invalid_dates"],
    "date_min": quality["date_min"],
    "date_max": quality["date_max"],
}
pd.Series(quality_summary, name="value")

In [ ]:
columns = pd.DataFrame(quality["column_info"])
columns[["name", "dtype", "missing", "missing_pct", "unique", "sample"]]

## Field mapping

Mapping confidence is reported on a 0–100 scale. A derived field is created from other mapped columns — for example, revenue can be derived as quantity × unit price when no revenue column exists.

In [ ]:
mapping = pd.DataFrame(field_mapping_view(ctx.mapping))
mapping[["field", "label", "column", "confidence", "method", "derived"]]

## Analysis capabilities

A capability is enabled only when all required fields are mapped. Missing capabilities are explained rather than filled with fabricated values.

In [ ]:
capabilities = pd.DataFrame(capabilities_view(ctx.mapping))
capabilities[["label", "available", "description"]]

In [ ]:
guidance = pd.DataFrame(quality["missing_data_guidance"])
guidance if not guidance.empty else "All core analysis fields are available."

In [ ]:
db.close()